**Import Libraries**

In [ ]:
import os
import glob
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pydicom
import cv2
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5
SAMPLE_SIZE = 3000

LABELS = [
    'any',
    'epidural',
    'intraparenchymal',
    'intraventricular',
    'subarachnoid',
    'subdural'
]

print("TensorFlow Version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

**Load RSNA Dataset**

In [ ]:
csv_files = glob.glob(
    '/kaggle/input/**/stage_2_train.csv',
    recursive=True
)

dcm_dirs = glob.glob(
    '/kaggle/input/**/stage_2_train',
    recursive=True
)

if not csv_files or not dcm_dirs:
    raise FileNotFoundError(
        "RSNA dataset attach karein. "
        "stage_2_train.csv aur stage_2_train folder required hain."
    )

csv_path = csv_files[0]

dcm_dir = next(
    p for p in dcm_dirs if os.path.isdir(p)
)

df = pd.read_csv(csv_path)

print("Dataset Shape:", df.shape)
display(df.head())

**Convert Labels into Multi-Label Format**

In [ ]:
df[['Image_ID', 'Type']] = df['ID'].str.rsplit(
    '_', n=1, expand=True
)

labels_df = df.pivot_table(
    index='Image_ID',
    columns='Type',
    values='Label',
    aggfunc='max'
)

labels_df = labels_df.reindex(
    columns=LABELS
).fillna(0).reset_index()

labels_df[LABELS] = labels_df[LABELS].astype('float32')

labels_df['path'] = labels_df['Image_ID'].apply(
    lambda x: os.path.join(dcm_dir, x + '.dcm')
)

labels_df = labels_df[
    labels_df['path'].apply(os.path.isfile)
].reset_index(drop=True)

print("Total Images:", len(labels_df))
display(labels_df.head())

**Dataset Analysis and Visualization**

In [ ]:
class_counts = labels_df[LABELS].sum()

plt.figure(figsize=(10, 5))

sns.barplot(
    x=class_counts.index,
    y=class_counts.values
)

plt.title("Distribution of Hemorrhage Classes")
plt.xlabel("Hemorrhage Type")
plt.ylabel("Number of Images")
plt.xticks(rotation=35)
plt.show()

print(class_counts)

**Select Dataset Sample**

In [ ]:
if len(labels_df) > SAMPLE_SIZE:
    labels_df, _ = train_test_split(
        labels_df,
        train_size=SAMPLE_SIZE,
        stratify=labels_df['any'],
        random_state=SEED
    )

labels_df = labels_df.reset_index(drop=True)

print("Selected Images:", len(labels_df))

**Split Dataset into Training, Validation and Testing**

In [ ]:
train_df, temp_df = train_test_split(
    labels_df,
    test_size=0.30,
    random_state=SEED,
    stratify=labels_df['any']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['any']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training Images:", len(train_df))
print("Validation Images:", len(val_df))
print("Testing Images:", len(test_df))

**DICOM Image Preprocessing and HU Windowing**

In [ ]:
def hu_window(image, center, width):
    lower = center - width / 2
    upper = center + width / 2

    image = np.clip(image, lower, upper)
    image = (image - lower) / (upper - lower)

    return image


def preprocess_dicom(path):
    dicom = pydicom.dcmread(path)

    image = dicom.pixel_array.astype(np.float32)

    slope = float(getattr(dicom, 'RescaleSlope', 1))
    intercept = float(getattr(dicom, 'RescaleIntercept', 0))

    image = image * slope + intercept

    # Three CT windows
    brain = hu_window(image, 40, 80)
    subdural = hu_window(image, 80, 200)
    bone = hu_window(image, 600, 2800)

    image = np.stack(
        [brain, subdural, bone],
        axis=-1
    )

    image = cv2.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    return image.astype(np.float32)

**Display Preprocessed CT Images**

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(6):
    image = preprocess_dicom(
        train_df.iloc[i]['path']
    )

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(f"CT Image {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

**Create Data Generator**

In [ ]:
def data_generator(dataframe, shuffle=False):

    paths = dataframe['path'].values
    labels = dataframe[LABELS].values.astype(np.float32)

    def generator():
        indices = np.arange(len(paths))

        if shuffle:
            np.random.shuffle(indices)

        for idx in indices:
            image = preprocess_dicom(paths[idx])
            yield image, labels[idx]

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(
                shape=(IMG_SIZE, IMG_SIZE, 3),
                dtype=tf.float32
            ),
            tf.TensorSpec(
                shape=(6,),
                dtype=tf.float32
            )
        )
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = data_generator(train_df, shuffle=True)
val_ds = data_generator(val_df)
test_ds = data_generator(test_df)

print("Datasets Ready!")

**Handle Class Imbalance**

In [ ]:
train_labels = train_df[LABELS].values

positive_counts = train_labels.sum(axis=0)
negative_counts = len(train_labels) - positive_counts

pos_weights = negative_counts / np.maximum(
    positive_counts, 1
)

pos_weights = np.clip(pos_weights, 1.0, 20.0)

print("Positive Class Weights:")
print(dict(zip(LABELS, pos_weights)))


def weighted_binary_crossentropy(y_true, y_pred):

    y_pred = tf.clip_by_value(
        y_pred, 1e-7, 1 - 1e-7
    )

    weights = tf.constant(
        pos_weights,
        dtype=tf.float32
    )

    loss = -(
        weights * y_true * tf.math.log(y_pred)
        + (1 - y_true) * tf.math.log(1 - y_pred)
    )

    return tf.reduce_mean(loss)

**Model 1 — Custom CNN**

In [ ]:
def build_custom_cnn():

    model = models.Sequential([

        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        layers.Conv2D(
            32, (3, 3),
            activation='relu',
            padding='same'
        ),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(
            64, (3, 3),
            activation='relu',
            padding='same'
        ),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(
            128, (3, 3),
            activation='relu',
            padding='same'
        ),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(6, activation='sigmoid')

    ])

    return model


custom_cnn = build_custom_cnn()
custom_cnn.summary()

**Model 2 — ResNet50 Transfer Learning**

In [ ]:
def build_resnet50():

    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    base_model.trainable = False

    inputs = layers.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    x = layers.Rescaling(255.0)(inputs)

    x = layers.Lambda(
        tf.keras.applications.resnet50.preprocess_input
    )(x)

    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        6, activation='sigmoid'
    )(x)

    model = models.Model(inputs, outputs)

    return model


resnet50 = build_resnet50()
resnet50.summary()

**Model 3 — MobileNetV2 Transfer Learning**

In [ ]:
def build_mobilenet():

    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    base_model.trainable = False

    inputs = layers.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Convert [0,1] pixels into [-1,1]
    x = layers.Rescaling(
        scale=2.0,
        offset=-1.0
    )(inputs)

    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        6, activation='sigmoid'
    )(x)

    model = models.Model(inputs, outputs)

    return model


mobilenet = build_mobilenet()
mobilenet.summary()

**Compile and Train All 3 Models**

In [ ]:
all_models = {
    "Custom CNN": custom_cnn,
    "ResNet50": resnet50,
    "MobileNetV2": mobilenet
}

histories = {}
training_times = {}

for name, model in all_models.items():

    print("\nTraining:", name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=weighted_binary_crossentropy,
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name='accuracy'
            )
        ]
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[early_stop]
    )

    training_times[name] = time.time() - start_time
    histories[name] = history.history

    model.save(
        f"/kaggle/working/{name.replace(' ', '_')}.keras"
    )

    print(name, "Training Completed!")

**Evaluate All Three Models**

In [ ]:
results = []
predictions = {}

y_true = test_df[LABELS].values.astype(int)

for name, model in all_models.items():

    print("\nEvaluating:", name)

    start_time = time.time()

    y_prob = model.predict(test_ds, verbose=0)

    inference_time = time.time() - start_time

    y_pred = (y_prob >= 0.5).astype(int)

    predictions[name] = y_prob

    accuracy = accuracy_score(
        y_true.flatten(),
        y_pred.flatten()
    )

    precision = precision_score(
        y_true, y_pred,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        y_true, y_pred,
        average='macro',
        zero_division=0
    )

    f1 = f1_score(
        y_true, y_pred,
        average='macro',
        zero_division=0
    )

    auc_scores = []

    for j in range(len(LABELS)):
        if len(np.unique(y_true[:, j])) == 2:
            auc_scores.append(
                roc_auc_score(
                    y_true[:, j],
                    y_prob[:, j]
                )
            )

    auc = np.mean(auc_scores) if auc_scores else np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": auc,
        "Training Time": training_times[name],
        "Inference Time": inference_time
    })

results_df = pd.DataFrame(results)

display(results_df)

**Classification Report and Confusion Matrix**

In [ ]:
for name, y_prob in predictions.items():

    print("\nModel:", name)

    y_pred = (y_prob >= 0.5).astype(int)

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=LABELS,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true[:, 0],
        y_pred[:, 0],
        labels=[0, 1]
    )

    plt.figure(figsize=(5, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['No Hemorrhage', 'Hemorrhage'],
        yticklabels=['No Hemorrhage', 'Hemorrhage']
    )

    plt.title(name + " - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

**Plot Training and Validation Accuracy**

In [ ]:
plt.figure(figsize=(12, 5))

for name, history in histories.items():

    plt.plot(
        history['accuracy'],
        label=name + " Train"
    )

    plt.plot(
        history['val_accuracy'],
        linestyle='--',
        label=name + " Validation"
    )

plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Binary Accuracy")
plt.legend()
plt.grid()
plt.show()

**Plot Training and Validation Loss**

In [ ]:
plt.figure(figsize=(12, 5))

for name, history in histories.items():

    plt.plot(
        history['loss'],
        label=name + " Train"
    )

    plt.plot(
        history['val_loss'],
        linestyle='--',
        label=name + " Validation"
    )

plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

**Compare All Three Models**

In [ ]:
metrics = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1 Score',
    'ROC AUC'
]

results_df.set_index('Model')[metrics].plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title("Comparison of Three Deep Learning Models")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

**Save Final Results**

In [ ]:
results_df.to_csv(
    '/kaggle/working/model_comparison.csv',
    index=False
)

print("All Models Trained and Evaluated!")
print("Results saved successfully.")

display(results_df)